# Phase 5 — Temporal Transformer
Water-System Anomaly Detection | Adaptive Temporal Transformer Project

Architecture: Input Projection → Sinusoidal PE → Transformer Encoder (2 layers) → Mean Pool → Classification Head
Config: d_model=32, nhead=2, layers=2, ff=64, dropout=0.1, lr=1e-3, batch=256, 5 epochs, stride=12

In [ ]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

metrics  = pd.read_csv('../results/transformer/final_metrics.csv')
exp_log  = pd.read_csv('../results/transformer/transformer_experiments.csv')
all_comp = pd.read_csv('../results/model_comparison/all_models.csv')
err      = pd.read_csv('../results/transformer/error_analysis.csv')

print('Transformer Test Metrics:')
print(metrics.to_string(index=False))
print('\nHP Search Results:')
print(exp_log[['experiment_id','d_model','nhead','num_layers','validation_f1','n_params']].to_string(index=False))

In [ ]:
print('All Models Comparison:')
print(all_comp.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, fname in zip(axes, ['loss_curve.png','confusion_matrix.png','roc_curve.png']):
    img = mpimg.imread(f'../figures/transformer/{fname}')
    ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Attention visualisation
import os
attn_files = [f for f in os.listdir('../figures/transformer/attention') if f.endswith('.png')]
if attn_files:
    fig, axes = plt.subplots(1, len(attn_files), figsize=(8*len(attn_files), 5))
    if len(attn_files) == 1: axes = [axes]
    for ax, fname in zip(axes, attn_files):
        img = mpimg.imread(f'../figures/transformer/attention/{fname}')
        ax.imshow(img); ax.axis('off'); ax.set_title(fname)
    plt.tight_layout(); plt.show()
else:
    print('No attention figures available.')

In [ ]:
img = mpimg.imread('../figures/model_comparison/all_models_metrics.png')
fig, ax = plt.subplots(figsize=(14, 5))
ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()

## Interpretation

The Transformer achieved F1=0.922, ROC-AUC=0.993 with only 5 epochs and 17,729 parameters.
This substantially outperforms the LSTM (F1=0.465) under identical hardware constraints.

**LSTM vs Transformer (same data, same epochs, same hardware):**
- Transformer F1=0.922 vs LSTM F1=0.465 — Transformer learns faster from limited epochs
- Transformer FNR=0.061 vs LSTM FNR=0.622 — Transformer misses far fewer anomalies
- Both have similar model size (~87 KB) and parameter count (~17-21K)
- Transformer inference is faster (0.70ms vs 1.10ms per sample)

**Transformer vs Classical Baselines:**
- RF (F1=0.978) and XGBoost (F1=0.978) still outperform the Transformer
- However, classical models used hand-crafted rolling statistics (64 features)
- The Transformer learned from raw 16-feature sequences with no feature engineering
- With more epochs and larger architecture, the Transformer is expected to close this gap

**Key limitation:** 5 epochs is insufficient for full convergence. The training F1 reached
0.95 while val F1 was 0.22 — indicating overfitting on the small stride=12 dataset.
The test F1=0.922 is higher than val F1 because the test set has more anomaly events
(20.3% vs 9.9%), making the low threshold (0.0198) effective there.